In [14]:
!mamba activate chem

zsh:1: command not found: mamba


In [13]:
import os
print(os.getcwd())

/Users/shreechatterjee/jak-selectivity-screening


In [1]:
import os
import csv
from collections import defaultdict
from pathlib import Path

In [3]:
import pandas as pd
import numpy as np

In [5]:
from rdkit import Chem
from rdkit.Chem import Draw

In [7]:
# manually define project root
PROJECT_ROOT = Path("/Users/shreechatterjee/jak-selectivity-screening")

# set working directory
os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/shreechatterjee/jak-selectivity-screening


In [9]:
!python src/protein_prep.py


=== JAK1 (Crystal Structures/JAK1_4EI4.cif) ===
    Detected binding-site ligand: 0Q2 (chain A, 20 atoms)
    Docking box center: (6.913700103759766, 57.60139846801758, 0.8156000375747681)
    --- Hetero residue audit for JAK1 ---
        PTR    chain A resseq  1034  ->  KEPT (whitelist)
        PTR    chain A resseq  1035  ->  KEPT (whitelist)
        0Q2    chain A resseq  1201  ->  REMOVED (not on keep-list)
        HOH    chain A resseq  1301  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1302  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1303  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1304  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1305  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1306  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1307  ->  REMOVED (explicit junk list)
        HOH    chain A resseq  1308  ->  REMOVED (explicit junk list)
        HOH    chain A r

In [ ]:
# data/ChEMBL/ChEMBL Data Preprocessing/ic50_shared_compounds.tsv

In [10]:
!python src/ligand_prep.py \
    --input "data/ChEMBL/ChEMBL Data Preprocessing/ic50_shared_compounds.tsv" \
    --output ligands_pdbqt.csv \
    --sep "\t"

/Users/shreechatterjee/jak-selectivity-screening/src/ligand_prep.py:138: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(args.input, sep=args.sep)
Preparing 5779 ligands from 'data/ChEMBL/ChEMBL Data Preprocessing/ic50_shared_compounds.tsv' ...
[RDKit] ERROR:[12:11:09] UFFTYPER: Unrecognized atom type: S_6+6 (22)
[RDKit] ERROR:[12:11:26] UFFTYPER: Unrecognized charge state for atom: 4
[RDKit] ERROR:[12:11:26] UFFTYPER: Unrecognized charge state for atom: 4
[RDKit] ERROR:[12:11:27] UFFTYPER: Unrecognized charge state for atom: 4
[RDKit] ERROR:[12:11:27] UFFTYPER: Unrecognized charge state for atom: 4
[!] Failed to prepare ligand 1328 (CCc1cc(O)ccc1-c1cc2[nH]ncc2c(N2CCc3c(cccc3NSC)C2)n1.O.O): RDKit molecule has 3 fragments. Must have 1.
[RDKit] ERROR:[12:13:45] UFFTYPER: Unrecogniz

In [18]:
ligands_pdbqt = pd.read_csv("docking/docking_prep/ligands_pdbqt.csv")

In [50]:
ligands_pdbqt.loc[1405]['PDBQT']

'REMARK SMILES CN(C)C1CCN(c2nc(-c3cnn4ccccc34)nc(N[C@H]3CC[C@H](CC#N)CC3)c2F)CC1\nREMARK SMILES IDX 8 1 9 2 10 3 20 4 21 5 32 6 33 7 7 8 4 9 5 10 35 11 6 12\nREMARK SMILES IDX 34 13 2 14 1 15 3 16 11 17 12 18 19 19 13 20 18 21 14 22\nREMARK SMILES IDX 17 23 15 24 16 25 22 26 23 28 31 29 24 30 30 31 25 32 26 33\nREMARK SMILES IDX 27 34 28 35 29 36\nREMARK H PARENT 22 27\nROOT\nATOM      1  C   UNL     1       1.369   0.115  -0.315  1.00  0.00     0.171 A \nATOM      2  N   UNL     1       1.244   1.374   0.150  1.00  0.00    -0.208 NA\nATOM      3  C   UNL     1       0.048   1.976   0.029  1.00  0.00     0.167 A \nATOM      4  N   UNL     1      -1.048   1.420  -0.508  1.00  0.00    -0.208 NA\nATOM      5  C   UNL     1      -0.925   0.170  -0.979  1.00  0.00     0.168 A \nATOM      6  C   UNL     1       0.283  -0.513  -0.923  1.00  0.00     0.207 A \nATOM      7  F   UNL     1       0.409  -1.735  -1.465  1.00  0.00    -0.199 F \nENDROOT\nBRANCH   1   8\nATOM      8  N   UNL     1   

In [17]:
os.path.exists(os.path.expanduser("~/bin/vina"))

True

In [12]:
one_selective_compound = ligands_pdbqt[ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"] < 1/200]

In [14]:
one_selective_compound

,smiles,JAK1_IC50_nM,JAK2_IC50_nM,matched_assay_description,ligand_id,PDBQT
1405,CN(C)C1CCN(c2nc(-c3cnn4ccccc34)nc(N[C@H]3CC[C@...,220.0,1.0,In Vitro JAK kinase Assays: The catalytic doma...,6c23b1a47748edc80d0778923112602a76e9f2560f5e70...,REMARK SMILES CN(C)C1CCN(c2nc(-c3cnn4ccccc34)n...


In [26]:
one_selective_compound.to_csv("one_selective_compound.csv")

In [16]:
jak2_selective = ligands_pdbqt[ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"] < 1/25]

In [20]:
jak2_selective.to_csv("jak2_selective.csv")

In [19]:
ligands_pdbqt

,smiles,JAK1_IC50_nM,JAK2_IC50_nM,matched_assay_description,ligand_id,PDBQT
0,BrC[C@H]1CC[C@H](c2nnn3cnc4[nH]ccc4c23)CC1,0.40,3.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",3f360bc33976e7fede2f0f742a6737260918e12448c515...,REMARK SMILES BrC[C@H]1CC[C@H](c2nnn3cnc4[nH]c...
1,Brc1cc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)sc1Br,21.00,71.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",0e72c6ed971acca15bf94462e4a6cefd6f04878e5060fb...,REMARK SMILES Brc1cc(CN2CCC(c3nnn4cnc5[nH]ccc5...
2,Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)cc1,1.10,6.1,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",b68d445d553a29323928b93fd7ccc4330911f3f9467e59...,REMARK SMILES Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc...
3,Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)o1,1.40,2.8,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",8edc22ec15b28b98d3ef8e4a6a3cb77252128025c85852...,REMARK SMILES Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc...
4,Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)s1,0.79,4.6,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",bd9aa38011da7cead211958fd54276c2aee2546c3fad9f...,REMARK SMILES Brc1ccc(CN2CCC(c3nnn4cnc5[nH]ccc...
...,...,...,...,...,...,...
5774,c1cncc(CN2CCCC(c3nnn4cnc5[nH]ccc5c34)C2)c1,210.00,620.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",914c8a1415ce10c30eef9ba20e8bb3b45bf79cb8652cb7...,REMARK SMILES c1cncc(CN2CCCC(c3nnn4cnc5[nH]ccc...
5775,c1coc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)c1,9.10,20.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",43e835f1a6fdc0b193742a36bf9bb245aa625b1ba148d3...,REMARK SMILES c1coc(CN2CCC(c3nnn4cnc5[nH]ccc5c...
5776,c1csc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)c1,4.30,10.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",c5a19e23c955d0667aafa52ead9a84e212adccacbb5502...,REMARK SMILES c1csc(CN2CCC(c3nnn4cnc5[nH]ccc5c...
5777,c1csc(CN2CCC(c3nnn4cnc5[nH]ccc5c34)CC2)n1,22.00,54.0,"Enzyme Assay: JAK1, JAK2, JAK3 and Tyk2 were p...",f7fb32393b94e2e48b77e803ab10b0dec0686c070402b2...,REMARK SMILES c1csc(CN2CCC(c3nnn4cnc5[nH]ccc5c...


In [11]:
jak2_10x = ligands_pdbqt[
    (ligands_pdbqt["JAK1_IC50_nM"] / ligands_pdbqt["JAK2_IC50_nM"]) >= 10
]

In [15]:
jak2_10x.to_csv("jak2_10x.csv")

In [17]:
jak2_5x = ligands_pdbqt[
    (ligands_pdbqt["JAK1_IC50_nM"] / ligands_pdbqt["JAK2_IC50_nM"]) >= 5
]
jak2_5x.to_csv("jak2_5x.csv")

In [20]:
jak1_5x = ligands_pdbqt[
    (ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"]) >= 5
]
jak1_5x.to_csv("jak1_5x.csv")

In [22]:
jak1_10x = ligands_pdbqt[
    (ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"]) >= 10
]
jak1_10x.to_csv("jak1_10x.csv")

In [24]:
nonselectives_5x_thresh = ligands_pdbqt[
    (ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"] < 5) &
    (ligands_pdbqt["JAK1_IC50_nM"] / ligands_pdbqt["JAK2_IC50_nM"] < 5)
]

nonselectives_5x_thresh.to_csv("nonselectives_5x_thresh.csv", index=False)

In [26]:
nonselectives_10x_thresh = ligands_pdbqt[
    (ligands_pdbqt["JAK2_IC50_nM"] / ligands_pdbqt["JAK1_IC50_nM"] < 10) &
    (ligands_pdbqt["JAK1_IC50_nM"] / ligands_pdbqt["JAK2_IC50_nM"] < 10)
]

nonselectives_10x_thresh.to_csv("nonselectives_10x_thresh.csv", index=False)